# MODUL 6 – Filter Spasial
## Low Pass Filter, High Pass Filter, Point Detection, Line Detection, Edge Detection

**Mata Kuliah:** Pengolahan Citra dan Visi Komputer  
**Nama:** [Alfreda Dhaifullah Mahezwara]  
**NIM:** [2441107020219]

## A. Persiapan: Import Library & Mount Google Drive

In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import math
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Mount drive dilewati:', e)

plt.rcParams['figure.figsize'] = (10, 5)
print('Library siap digunakan.')

## B. Load Citra
Ubah path di bawah sesuai lokasi citra di Google Drive Anda. Jika file tidak ditemukan, kode akan menggunakan citra contoh dari internet.

In [ ]:
image_path = '/content/WhatsApp Image 2026-09-30 at 13.06.03.jpeg'  # <-- GANTI dengan path citra Anda

def load_image(path):
    if os.path.exists(path):
        img = cv.imread(path)
        if img is None:
            raise ValueError('File tidak dapat dibaca: ' + path)
        return img
    print('File tidak ditemukan, menggunakan citra contoh dari URL...')
    import urllib.request
    url = 'https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png'
    urllib.request.urlretrieve(url, '/tmp/sample.png')
    return cv.imread('/tmp/sample.png')

img_bgr = load_image(image_path)
img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

plt.imshow(img_rgb)
plt.title('Citra Input')
plt.axis('off')
plt.show()

In [ ]:
def show_side_by_side(img1, img2, title1='Citra Asli', title2='Hasil Filter', gray=True):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img1, cmap='gray' if gray else None)
    plt.title(title1); plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(img2, cmap='gray' if gray else None)
    plt.title(title2); plt.axis('off')
    plt.tight_layout()
    plt.show()

## C. Praktikum Konvolusi (Fungsi Sendiri, Tanpa Library)

In [ ]:
def konvolusi_manual(image, kernel):
    """Fungsi konvolusi spasial sederhana (grayscale) dengan padding 0.
    image: citra grayscale (H, W), kernel: matriks (kh, kw)"""
    H, W = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    result = np.zeros((H, W), dtype=np.float64)
    for i in range(H):
        for j in range(W):
            region = padded[i:i+kh, j:j+kw]
            result[i, j] = np.sum(region * kernel)
    return np.clip(result, 0, 255).astype(np.uint8)

# Contoh kernel sharpening
kernel_sharpen = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])

hasil_sharpen_manual = konvolusi_manual(img_gray, kernel_sharpen)
show_side_by_side(img_gray, hasil_sharpen_manual, 'Asli', 'Sharpening (konvolusi manual)')

## D. Definisi Kernel-Kernel Filter

In [ ]:
# Average Filter 3x3
kernel_average = np.ones((3, 3), dtype=np.float32) / 9

# Low Pass Filter
kernel_lowpass = np.array([[1, 1, 1],
                           [1, 4, 1],
                           [1, 1, 1]], dtype=np.float32) / 12

# High Pass Filter
kernel_highpass = np.array([[-1, -1, -1],
                            [-1,  8, -1],
                            [-1, -1, -1]], dtype=np.float32)

# Point Detection
kernel_point = np.array([[-1, -1, -1],
                         [-1,  8, -1],
                         [-1, -1, -1]], dtype=np.float32)

# Line Detection (horizontal & vertikal)
kernel_line_h = np.array([[-1, -1, -1],
                          [ 2,  2,  2],
                          [-1, -1, -1]], dtype=np.float32)
kernel_line_v = np.array([[-1,  2, -1],
                          [-1,  2, -1],
                          [-1,  2, -1]], dtype=np.float32)

# Edge Detection (Sobel)
kernel_sobel_x = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]], dtype=np.float32)
kernel_sobel_y = np.array([[-1, -2, -1],
                           [ 0,  0,  0],
                           [ 1,  2,  1]], dtype=np.float32)

# Emboss
kernel_emboss = np.array([[-2, -1, 0],
                          [-1,  1, 1],
                          [ 0,  1, 2]], dtype=np.float32)

# Sobel Left Edge
kernel_sobel_left = np.array([[1, 0, -1],
                              [2, 0, -2],
                              [1, 0, -1]], dtype=np.float32)

# Canny-like kernel
kernel_canny = np.array([[-1, -1, -1],
                         [-1,  8, -1],
                         [-1, -1, -1]], dtype=np.float32)

# Gaussian Kernel 21x21
kernel_size = 21
sigma = math.sqrt(kernel_size)
gaussian_kernel_1d = cv.getGaussianKernel(kernel_size, sigma)
kernel_gaussian = gaussian_kernel_1d @ gaussian_kernel_1d.T

print('Kernel Gaussian 21x21 terbentuk, shape:', kernel_gaussian.shape)

## E. Penerapan Berbagai Filter pada Citra Keabuan (Konvolusi Manual)

In [ ]:
hasil = {}
hasil['Average'] = cv.filter2D(img_gray, -1, kernel_average)
hasil['Low Pass'] = cv.filter2D(img_gray, -1, kernel_lowpass)
hasil['High Pass'] = cv.filter2D(img_gray, -1, kernel_highpass)
hasil['Point'] = cv.filter2D(img_gray, -1, kernel_point)
hasil['Line H'] = cv.filter2D(img_gray, -1, kernel_line_h)
hasil['Line V'] = cv.filter2D(img_gray, -1, kernel_line_v)
hasil['Emboss'] = cv.filter2D(img_gray, -1, kernel_emboss)
hasil['Sobel Left'] = cv.filter2D(img_gray, -1, kernel_sobel_left)
hasil['Gaussian 21x21'] = cv.filter2D(img_gray, -1, kernel_gaussian)

for nama, h in hasil.items():
    show_side_by_side(img_gray, h, 'Asli', nama)

## F. Percobaan 1: Filter Gaussian, Sharpen, Canny dengan `filter2D` (Citra RGB)

In [ ]:
gauss_rgb = cv.filter2D(img_rgb, -1, kernel_gaussian)
sharpen_rgb = cv.filter2D(img_rgb, -1, kernel_sharpen)
canny_like = cv.filter2D(img_rgb, -1, kernel_canny)

plt.figure(figsize=(16, 5))
for i, (t, im) in enumerate([('Asli', img_rgb), ('Gaussian', gauss_rgb), ('Sharpen', sharpen_rgb), ('Canny-like', canny_like)]):
    plt.subplot(1, 4, i+1); plt.imshow(np.clip(im,0,255).astype(np.uint8)); plt.title(t); plt.axis('off')
plt.show()

## G. Percobaan 2: Bilateral Filter dan Guided Filter

In [ ]:
bilateral = cv.bilateralFilter(img_rgb, d=9, sigmaColor=75, sigmaSpace=75)

def guided_filter(guide, src, r=8, eps=1e-2):
    guide = guide.astype(np.float64) / 255.0
    src = src.astype(np.float64) / 255.0
    ksize = (2*r + 1, 2*r + 1)
    mean_I = cv.boxFilter(guide, -1, ksize)
    mean_p = cv.boxFilter(src, -1, ksize)
    mean_Ip = cv.boxFilter(guide * src, -1, ksize)
    cov_Ip = mean_Ip - mean_I * mean_p
    mean_II = cv.boxFilter(guide * guide, -1, ksize)
    var_I = mean_II - mean_I * mean_I
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    mean_a = cv.boxFilter(a, -1, ksize)
    mean_b = cv.boxFilter(b, -1, ksize)
    q = mean_a * guide + mean_b
    return np.clip(q * 255, 0, 255).astype(np.uint8)

guide_img = cv.cvtColor(img_rgb, cv.COLOR_RGB2GRAY)
guided = guided_filter(guide_img, cv.cvtColor(img_rgb, cv.COLOR_RGB2GRAY))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(bilateral); plt.title('Bilateral Filter'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(guided, cmap='gray'); plt.title('Guided Filter'); plt.axis('off')
plt.show()

## H. Percobaan 3: Feature Map pada CNN (Convolution di Deep Learning)
Jalankan beberapa kali untuk melihat variasi hasil (kernel conv bersifat random).

In [ ]:
try:
    import torch
    import torch.nn as nn

    tensor = torch.from_numpy(img_gray.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0)
    conv = nn.Conv2d(1, 6, kernel_size=5, padding=2)  # 6 kernel filter random
    with torch.no_grad():
        feature_maps = conv(tensor)

    plt.figure(figsize=(18, 4))
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(feature_maps[0, i].numpy(), cmap='gray')
        plt.title(f'Feature Map {i+1}'); plt.axis('off')
    plt.show()
    print('Kesimpulan: setiap kernel konvolusi menghasilkan respons tepi/tekstur yang berbeda pada feature map.')
except ImportError:
    print('PyTorch tidak tersedia.')

## I. Percobaan 4: Efek Beauty dan Vintage (Kombinasi Filter Tradisional)

In [ ]:
# --- Beauty: bilateral smoothing + sedikit pencerahan ---
beauty = cv.bilateralFilter(img_rgb, 15, 80, 80)
beauty = cv.addWeighted(beauty, 0.8, img_rgb, 0.2, 20)

# --- Vintage: sepia + vignette + saturasi turun ---
sepia_kernel = np.array([[0.272, 0.534, 0.131],
                         [0.349, 0.686, 0.168],
                         [0.393, 0.769, 0.189]])
vintage = cv.transform(img_rgb.astype(np.float64), sepia_kernel.T)
vintage = np.clip(vintage, 0, 255).astype(np.uint8)
h, w = img_rgb.shape[:2]
Y, X = np.ogrid[:h, :w]
mask = 1 - (((X - w/2)**2)/(w/2)**2 + ((Y - h/2)**2)/(h/2)**2)/2
mask = np.clip(mask, 0.3, 1)
vintage = np.clip(vintage * mask[..., None], 0, 255).astype(np.uint8)

show_side_by_side(img_rgb, beauty, 'Asli', 'Beauty', gray=False)
show_side_by_side(img_rgb, vintage, 'Asli', 'Vintage', gray=False)

## J. Percobaan 5: Efek Kartun/Anime (Bilateral + Edge)

In [ ]:
def kartun(image):
    smooth = cv.bilateralFilter(image, 9, 200, 200)
    smooth = cv.medianBlur(smooth, 7)
    gray = cv.cvtColor(smooth, cv.COLOR_RGB2GRAY)
    edges = cv.adaptiveThreshold(gray, 255, cv.ADAPTIVE_THRESH_MEAN_C,
                                 cv.THRESH_BINARY, 9, 2)
    edges_rgb = cv.cvtColor(edges, cv.COLOR_GRAY2RGB)
    return cv.bitwise_and(smooth, smooth, mask=edges)

cartoon = kartun(img_rgb)
show_side_by_side(img_rgb, cartoon, 'Asli', 'Efek Kartun', gray=False)

## K. Percobaan 6: Efek Filter Malam

In [ ]:
night = (img_rgb.astype(np.float32) * np.array([0.4, 0.4, 0.7])).astype(np.uint8)
night = cv.GaussianBlur(night, (5, 5), 0)
show_side_by_side(img_rgb, night, 'Asli', 'Filter Malam', gray=False)

## L. Percobaan 7: Filter Pagi dan Pagi + Kabut

In [ ]:
morning = cv.convertScaleAbs(img_rgb, alpha=1.1, beta=30)
morning = np.clip(morning.astype(np.int16) + np.array([20, 10, -10]), 0, 255).astype(np.uint8)

fog = cv.addWeighted(morning, 0.6, 255*np.ones_like(morning), 0.4, 0).astype(np.uint8)

show_side_by_side(img_rgb, morning, 'Asli', 'Filter Pagi', gray=False)
show_side_by_side(img_rgb, fog, 'Asli', 'Pagi + Kabut', gray=False)

## M. Tugas 1: Mean Filter vs Median Filter pada Noise Salt & Pepper

In [ ]:
def add_salt_pepper(img, amount=0.05):
    out = img.copy()
    n = int(amount * img.size)
    coords = [np.random.randint(0, i - 1, n) for i in img.shape]
    out[coords[0], coords[1]] = 255
    coords = [np.random.randint(0, i - 1, n) for i in img.shape]
    out[coords[0], coords[1]] = 0
    return out

noisy = add_salt_pepper(img_gray, 0.05)
mean_filtered = cv.blur(noisy, (3, 3))
median_filtered = cv.medianBlur(noisy, 3)

plt.figure(figsize=(16, 4))
for i, (t, im) in enumerate([('Noise', noisy), ('Mean 3x3', mean_filtered), ('Median 3x3', median_filtered)]):
    plt.subplot(1, 3, i+1); plt.imshow(im, cmap='gray'); plt.title(t); plt.axis('off')
plt.show()
print('Analisis: Median filter lebih bersih karena mengambil nilai median, menghilangkan titik ekstrem noise; mean filter meratakan sehingga noise masih terlihat samar.')

## N. Tugas 2: Kernel Sharpening 3x3 vs 5x5

In [ ]:
blurry = cv.GaussianBlur(img_gray, (7, 7), 0)

k3 = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
k5 = np.array([[0, 0, -1, 0, 0],
               [0, 0, -1, 0, 0],
               [-1,-1, 9,-1,-1],
               [0, 0, -1, 0, 0],
               [0, 0, -1, 0, 0]])

sharp3 = cv.filter2D(blurry, -1, k3)
sharp5 = cv.filter2D(blurry, -1, k5)

plt.figure(figsize=(16, 4))
for i, (t, im) in enumerate([('Blurry', blurry), ('Sharpen 3x3', sharp3), ('Sharpen 5x5', sharp5)]):
    plt.subplot(1, 3, i+1); plt.imshow(im, cmap='gray'); plt.title(t); plt.axis('off')
plt.show()
print('Evaluasi: 3x3 lebih halus & noise rendah; 5x5 tepi lebih tajam tetapi noise lebih tinggi. Kernel 3x3 umumnya optimal untuk penggunaan harian.')

## O. Tugas 3: Filter Kartun Sederhana (sudah pada Percobaan 5)
Fungsi `kartun(image)` menggabungkan bilateral filter (penghalusan warna), Canny/adaptive threshold (deteksi tepi), dan penggabungan via operasi piksel.

In [ ]:
# Demonstrasi ulang fungsi kartun pada RGB
cartoon_final = kartun(img_rgb)
show_side_by_side(img_rgb, cartoon_final, 'Citra Asli', 'Efek Kartun', gray=False)